# Поиск статей Авито: lexical + dense retrieval + reranker

Решение сравнивает три уровня поиска:

1. **lexical retrieval** — TF-IDF и BM25 ищут совпадающие слова;
2. **dense retrieval** — multilingual E5 ищет похожие по смыслу фрагменты;
3. **cross-encoder reranker** — внимательно оценивает пары «запрос — кандидат» и переставляет найденные статьи.

Все вычисления выполняются локально. `test.f` используется только после сравнения методов на `calibration.f`.

## 1. Настройки и данные

Используются две открытые локальные модели:

- `intfloat/multilingual-e5-small` — компактная multilingual retrieval-модель, лицензия MIT;
- `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` — multilingual text-ranking модель, лицензия Apache-2.0.

`local_files_only=True` запрещает удалённый inference: Notebook читает заранее скачанные веса из локального cache.

In [70]:
from functools import lru_cache
from pathlib import Path
import gc
import os
import random
import re

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
from bs4 import BeautifulSoup
from IPython.display import display
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

DENSE_MODEL_NAME = "intfloat/multilingual-e5-small"
DENSE_MODEL_REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
RERANKER_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
RERANKER_REVISION = "1427fd652930e4ba29e8149678df786c240d8825"

articles = pd.read_feather(DATA_DIR / "articles.f")
calibration = pd.read_feather(DATA_DIR / "calibration.f")
test = pd.read_feather(DATA_DIR / "test.f")

assert articles.columns.tolist() == ["article_id", "title", "body"]
assert calibration.columns.tolist() == ["query_id", "query_text", "ground_truth"]
assert test.columns.tolist() == ["query_id", "query_text"]
assert not articles["article_id"].duplicated().any()
assert not calibration["query_id"].duplicated().any()
assert not test["query_id"].duplicated().any()
print(f"Статьи: {len(articles)}, calibration: {len(calibration)}, test: {len(test)}")

Статьи: 793, calibration: 500, test: 500


## 2. Очистка и стемминг

HTML превращается в видимые текстовые блоки. Точные повторы длинных блоков удаляются. Стемминг используется только в BM25.

In [71]:
def normalize_text(text):
    return re.sub(r"\s+", " ", str(text).lower().replace("ё", "е")).strip()


def html_to_blocks(html):
    soup = BeautifulSoup(str(html), "html.parser")
    for tag in soup(["script", "style", "noscript", "template", "svg"]):
        tag.decompose()
    result, seen = [], set()
    for raw_block in soup.stripped_strings:
        block = normalize_text(raw_block)
        key = re.sub(r"\W+", " ", block).strip()
        if len(key) >= 40 and key in seen:
            continue
        result.append(block)
        if len(key) >= 40:
            seen.add(key)
    return result


TOKEN_RE = re.compile(r"[a-zа-я0-9]+")
stemmer = SnowballStemmer("russian")


@lru_cache(maxsize=100_000)
def stem_token(token):
    return stemmer.stem(token)


def stem_text(text):
    return " ".join(stem_token(token) for token in TOKEN_RE.findall(text))


articles["title_clean"] = articles["title"].map(normalize_text)
articles["body_blocks"] = articles["body"].map(html_to_blocks)
articles["body_clean"] = articles["body_blocks"].str.join(" ")
articles["title_stem"] = articles["title_clean"].map(stem_text)
articles["body_stem"] = articles["body_clean"].map(stem_text)
calibration["query_clean"] = calibration["query_text"].map(normalize_text)
test["query_clean"] = test["query_text"].map(normalize_text)
calibration["query_stem"] = calibration["query_clean"].map(stem_text)
test["query_stem"] = test["query_clean"].map(stem_text)
calibration["relevant_ids"] = calibration["ground_truth"].map(
    lambda value: [int(item) for item in value.split()]
)

duplicate = "достаточно длинный повторяющийся блок для автоматической проверки очистки"
assert html_to_blocks(f"<p>{duplicate}</p><div>{duplicate}</div>") == [duplicate]

## 3. Метрики и пять folds

- **MAP@10** учитывает наличие и позиции правильных статей в top-10.
- **Recall@k** показывает, какую долю правильных статей мы нашли среди первых `k`.
- Пять folds показывают, насколько результат стабилен на разных частях `calibration.f`.

In [72]:
def average_precision_at_k(predicted, relevant, k=10):
    relevant, seen = set(relevant), set()
    hits = 0
    score = 0.0
    for rank, item in enumerate(predicted[:k], 1):
        if item in relevant and item not in seen:
            hits += 1
            score += hits / rank
        seen.add(item)
    return score / min(len(relevant), k) if relevant else 0.0


def recall_at_k(predicted, relevant, k):
    return len(set(predicted[:k]) & set(relevant)) / len(set(relevant))


def rank_articles(scores, k=50):
    order = np.argsort(-scores, axis=1, kind="stable")[:, :k]
    return articles["article_id"].to_numpy()[order]


truth = calibration["relevant_ids"].tolist()
folds = list(StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE).split(
    calibration, calibration["relevant_ids"].str.len().clip(upper=3)
))
assert sorted(np.concatenate([validation for _, validation in folds]).tolist()) == list(range(len(calibration)))

## 4. Lexical retrieval

Word TF-IDF, character TF-IDF и BM25 строят отдельные рейтинги. RRF объединяет их без смешивания несопоставимых шкал score.

In [73]:
TITLE_WEIGHT = 0.10


def fit_tfidf(analyzer):
    models, matrices = {}, {}
    for field in ("title", "body"):
        vectorizer = (
            TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True,
                            max_features=180_000, dtype=np.float32)
            if analyzer == "word" else
            TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                            sublinear_tf=True, max_features=200_000, dtype=np.float32)
        )
        matrices[field] = vectorizer.fit_transform(articles[f"{field}_clean"])
        models[field] = vectorizer
    return models, matrices


def tfidf_scores(models, matrices, queries):
    parts = {
        field: (models[field].transform(queries) @ matrices[field].T).toarray()
        for field in ("title", "body")
    }
    return TITLE_WEIGHT * parts["title"] + (1 - TITLE_WEIGHT) * parts["body"]


def fit_bm25(documents, k1=1.5, b=0.75):
    vectorizer = CountVectorizer()
    counts = vectorizer.fit_transform(documents).astype(np.float32)
    lengths = np.asarray(counts.sum(axis=1)).ravel()
    df = np.asarray((counts > 0).sum(axis=0)).ravel()
    idf = np.log1p((len(documents) - df + 0.5) / (df + 0.5)).astype(np.float32)
    rows = np.repeat(np.arange(len(documents)), np.diff(counts.indptr))
    tf = counts.data
    counts.data = (
        tf * (k1 + 1)
        / (tf + k1 * (1 - b + b * lengths[rows] / lengths.mean()))
        * idf[counts.indices]
    )
    return vectorizer, counts


def bm25_scores(vectorizer, matrix, queries):
    query_matrix = vectorizer.transform(queries).astype(bool).astype(np.float32)
    return (query_matrix @ matrix.T).toarray()


def row_normalize(scores):
    return scores / np.maximum(scores.max(axis=1, keepdims=True), 1e-9)


def rrf(score_matrices, weights, k=20):
    fused = np.zeros_like(score_matrices[0], dtype=np.float32)
    rows = np.arange(len(fused))[:, None]
    for weight, scores in zip(weights, score_matrices):
        order = np.argsort(-scores, axis=1, kind="stable")
        ranks = np.empty_like(order)
        ranks[rows, order] = np.arange(1, scores.shape[1] + 1)
        fused += weight / (k + ranks)
    return fused


word_models, word_matrices = fit_tfidf("word")
char_models, char_matrices = fit_tfidf("char")
bm25_models, bm25_matrices = {}, {}
for field in ("title", "body"):
    bm25_models[field], bm25_matrices[field] = fit_bm25(articles[f"{field}_stem"])


def lexical_scores(clean_queries, stem_queries):
    word = tfidf_scores(word_models, word_matrices, clean_queries)
    char = tfidf_scores(char_models, char_matrices, clean_queries)
    bm25_parts = {
        field: bm25_scores(bm25_models[field], bm25_matrices[field], stem_queries)
        for field in ("title", "body")
    }
    bm25 = (
        TITLE_WEIGHT * row_normalize(bm25_parts["title"])
        + (1 - TITLE_WEIGHT) * row_normalize(bm25_parts["body"])
    )
    return rrf([word, char, bm25], [1, 1, 2])


lexical_calibration = lexical_scores(calibration["query_clean"], calibration["query_stem"])
lexical_test = lexical_scores(test["query_clean"], test["query_stem"])

## 5. Dense retrieval: multilingual E5

Статьи разбиваются на chunks. E5 кодирует запросы и chunks с предусмотренными авторами префиксами `query:` и `passage:`. Для статьи берётся score её наиболее похожего фрагмента.

In [74]:
dense_tokenizer = AutoTokenizer.from_pretrained(
    DENSE_MODEL_NAME, revision=DENSE_MODEL_REVISION, local_files_only=True
)
dense_model = AutoModel.from_pretrained(
    DENSE_MODEL_NAME, revision=DENSE_MODEL_REVISION, local_files_only=True
).eval()


def article_chunks(title, blocks, chunk_tokens=192, overlap=24):
    title_ids = dense_tokenizer.encode(title, add_special_tokens=False)[:48]
    body_ids = []
    for block in blocks:
        body_ids.extend(dense_tokenizer.encode(block, add_special_tokens=False))
        body_ids.append(dense_tokenizer.sep_token_id)
    payload = chunk_tokens - len(title_ids) - 1
    step = max(1, payload - overlap)
    return [
        dense_tokenizer.decode(title_ids + [dense_tokenizer.sep_token_id] + body_ids[start:start + payload])
        for start in range(0, max(1, len(body_ids)), step)
    ]


chunks, chunk_article_indices = [], []
for article_index, row in articles.iterrows():
    parts = article_chunks(row["title_clean"], row["body_blocks"])
    chunks.extend(parts)
    chunk_article_indices.extend([article_index] * len(parts))
chunk_article_indices = np.asarray(chunk_article_indices)


def mean_pool(hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1)
    return (hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)


def embed(texts, prefix, batch_size=32):
    result = []
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            prepared = [prefix + text for text in texts[start:start + batch_size]]
            batch = dense_tokenizer(
                prepared, padding=True, truncation=True, max_length=192, return_tensors="pt"
            )
            output = dense_model(**batch).last_hidden_state
            vectors = mean_pool(output, batch["attention_mask"])
            result.append(torch.nn.functional.normalize(vectors).cpu().numpy())
    return np.vstack(result)


chunk_embeddings = embed(chunks, "passage: ")


def dense_article_scores(queries):
    similarities = embed(list(queries), "query: ") @ chunk_embeddings.T
    scores = np.empty((len(queries), len(articles)), dtype=np.float32)
    best_chunks = np.empty((len(queries), len(articles)), dtype=np.int32)
    for article_index in range(len(articles)):
        positions = np.flatnonzero(chunk_article_indices == article_index)
        local_scores = similarities[:, positions]
        local_best = local_scores.argmax(axis=1)
        scores[:, article_index] = local_scores[np.arange(len(queries)), local_best]
        best_chunks[:, article_index] = positions[local_best]
    return scores, best_chunks


dense_calibration, best_calibration_chunks = dense_article_scores(calibration["query_clean"])
dense_test, best_test_chunks = dense_article_scores(test["query_clean"])
retrieval_calibration = rrf([lexical_calibration, dense_calibration], [2, 1])
retrieval_test = rrf([lexical_test, dense_test], [2, 1])

print(
    f"Фрагментов: {len(chunks)}, embedding shape: {chunk_embeddings.shape}, "
    f"параметров E5: {sum(p.numel() for p in dense_model.parameters()) / 1e6:.1f}M"
)
del dense_model, chunk_embeddings
gc.collect()

Фрагментов: 6967, embedding shape: (6967, 384), параметров E5: 117.7M


19336

## 6. Локальный cross-encoder reranker

Для каждого запроса объединяются top-15 lexical и top-15 dense статей. Reranker получает запрос и лучший найденный chunk каждого кандидата, после чего создаёт более точный рейтинг. Умеренный candidate pool сохраняет двухэтапную архитектуру, но заметно сокращает CPU-время.

В отличие от embedding-модели, cross-encoder читает запрос и текст совместно — поэтому он медленнее, но лучше подходит для окончательной сортировки небольшого списка.

In [75]:
reranker_tokenizer = AutoTokenizer.from_pretrained(
    RERANKER_NAME, revision=RERANKER_REVISION, local_files_only=True
)
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_NAME, revision=RERANKER_REVISION, local_files_only=True
).eval()


def candidate_article_indices(lexical, dense, top_n=15):
    result = []
    for lexical_row, dense_row in zip(lexical, dense):
        lexical_top = np.argsort(-lexical_row, kind="stable")[:top_n]
        dense_top = np.argsort(-dense_row, kind="stable")[:top_n]
        result.append(np.asarray(list(dict.fromkeys([*lexical_top, *dense_top])), dtype=np.int32))
    return result


def reranker_scores(queries, candidates, best_chunks, batch_size=64):
    scores = np.full((len(queries), len(articles)), -1e9, dtype=np.float32)
    pairs, locations = [], []
    for query_index, article_indices in enumerate(candidates):
        for article_index in article_indices:
            pairs.append((queries.iloc[query_index], chunks[best_chunks[query_index, article_index]]))
            locations.append((query_index, article_index))

    with torch.inference_mode():
        for start in range(0, len(pairs), batch_size):
            batch_pairs = pairs[start:start + batch_size]
            batch = reranker_tokenizer(
                [pair[0] for pair in batch_pairs],
                [pair[1] for pair in batch_pairs],
                padding=True, truncation=True, max_length=160, return_tensors="pt",
            )
            batch_scores = reranker_model(**batch).logits.view(-1).cpu().numpy()
            for (query_index, article_index), score in zip(
                locations[start:start + batch_size], batch_scores
            ):
                scores[query_index, article_index] = score
    return scores


calibration_candidates = candidate_article_indices(
    lexical_calibration, dense_calibration
)
reranker_calibration = reranker_scores(
    calibration["query_clean"], calibration_candidates, best_calibration_chunks
)
final_calibration = rrf(
    [lexical_calibration, dense_calibration, reranker_calibration], [2, 1, 2]
)

print(
    f"Среднее число кандидатов: {np.mean([len(items) for items in calibration_candidates]):.1f}; "
    f"параметров reranker: {sum(p.numel() for p in reranker_model.parameters()) / 1e6:.1f}M"
)
del reranker_model
gc.collect()

Среднее число кандидатов: 24.8; параметров reranker: 117.6M


0

## 7. Сравнение методов

Основной критерий — средний 5-fold MAP@10. Candidate recall показывает верхний предел reranker: если правильной статьи нет среди кандидатов, пересортировка вернуть её не сможет.

In [76]:
def evaluate(name, scores):
    predictions = rank_articles(scores, 50)
    fold_map = [
        np.mean([average_precision_at_k(predictions[i], truth[i]) for i in validation])
        for _, validation in folds
    ]
    return {
        "method": name,
        **{f"fold {i}": value for i, value in enumerate(fold_map, 1)},
        "MAP@10 mean": np.mean(fold_map),
        "MAP@10 std": np.std(fold_map),
        **{
            f"Recall@{k}": np.mean([
                recall_at_k(predictions[i], truth[i], k) for i in range(len(truth))
            ])
            for k in (10, 20, 50)
        },
    }


calibration_methods = {
    "lexical": lexical_calibration,
    "dense: multilingual-e5-small": dense_calibration,
    "RRF: 2×lexical + dense": retrieval_calibration,
    "cross-encoder reranker": reranker_calibration,
    "RRF: 2×lexical + dense + 2×reranker": final_calibration,
}
test_methods = {
    "lexical": lexical_test,
    "dense: multilingual-e5-small": dense_test,
    "RRF: 2×lexical + dense": retrieval_test,
}
comparison = pd.DataFrame([
    evaluate(name, scores) for name, scores in calibration_methods.items()
]).sort_values("MAP@10 mean", ascending=False, ignore_index=True)

candidate_recall = np.mean([
    len(set(articles.iloc[candidates]["article_id"]) & set(truth[index])) / len(set(truth[index]))
    for index, candidates in enumerate(calibration_candidates)
])
numeric_columns = comparison.columns.drop("method")
display(comparison.style.format({column: "{:.6f}" for column in numeric_columns}))
print(f"Candidate recall: {candidate_recall:.6f}")

best_method = comparison.loc[0, "method"]
best_map10 = float(comparison.loc[0, "MAP@10 mean"])
lexical_map10 = float(comparison.loc[comparison["method"] == "lexical", "MAP@10 mean"].iloc[0])
print(f"Победитель: {best_method}")
print(f"Прирост против lexical: {best_map10 - lexical_map10:+.6f}")

,method,fold 1,fold 2,fold 3,fold 4,fold 5,MAP@10 mean,MAP@10 std,Recall@10,Recall@20,Recall@50
0,RRF: 2×lexical + dense,0.453812,0.428544,0.391455,0.375709,0.380609,0.406026,0.030226,0.747000,0.854500,0.958000
1,RRF: 2×lexical + dense + 2×reranker,0.419484,0.408483,0.396300,0.347192,0.391196,0.392531,0.024708,0.736500,0.858500,0.948333
2,lexical,0.382054,0.308708,0.302627,0.323944,0.266121,0.316691,0.037799,0.652000,0.801167,0.930000
3,dense: multilingual-e5-small,0.247048,0.320939,0.280525,0.279623,0.287393,0.283106,0.023532,0.590667,0.759500,0.907000
4,cross-encoder reranker,0.317894,0.272351,0.301476,0.240136,0.255981,0.277568,0.028608,0.579500,0.786167,0.876500


Candidate recall: 0.876500
Победитель: RRF: 2×lexical + dense
Прирост против lexical: +0.089335


## 8. Анализ изменений

Показываем запросы, для которых победивший pipeline сильнее всего улучшил или ухудшил AP@10 относительно lexical baseline.

In [77]:
lexical_predictions = rank_articles(lexical_calibration, 10)
best_predictions = rank_articles(calibration_methods[best_method], 10)
deltas = pd.DataFrame({
    "query": calibration["query_text"],
    "relevant": truth,
    "lexical AP@10": [
        average_precision_at_k(lexical_predictions[i], truth[i]) for i in range(len(truth))
    ],
    "final AP@10": [
        average_precision_at_k(best_predictions[i], truth[i]) for i in range(len(truth))
    ],
})
deltas["delta"] = deltas["final AP@10"] - deltas["lexical AP@10"]
deltas["final top-10"] = best_predictions.tolist()
display(pd.concat([deltas.nlargest(3, "delta"), deltas.nsmallest(3, "delta")]))

,query,relevant,lexical AP@10,final AP@10,delta,final top-10
91,"Здравствуйте,как работаем авито доставка? Должна ли я что-то платить отправляя заказ?",[4234],0.125000,1.000000,0.875000,"[4234, 4532, 4361, 1958, 4286, 4308, 4513, 1909, 1901, 4208]"
455,почему высокая цена на доставку,[1951],0.166667,1.000000,0.833333,"[1951, 2232, 4321, 4319, 2911, 2948, 4320, 3128, 4436, 1901]"
369,"Просят подключить доставку 5ки, но как быть с безопасностью товара?",[4234],0.200000,1.000000,0.800000,"[4234, 4045, 2698, 1910, 4153, 4446, 4050, 1901, 4362, 4331]"
391,"Отмените платеж за доставку, чтобы я мог забрать товар, который покупатель не забрал",[4403],1.000000,0.142857,-0.857143,"[4308, 2943, 4361, 4532, 4234, 2408, 4403, 4387, 2962, 4219]"
41,Я хочу сделать заказ. В графе доставка лучшая цена указано авито <MONEY> А при оформлении цена авито доставки <MONEY> Можно узнать почему.,[1951],1.000000,0.333333,-0.666667,"[4320, 4308, 1951, 4234, 4214, 2962, 4286, 2646, 1909, 4361]"
174,"Не приходит оплата за продажи, после перехода на тариф с комиссией за продажу не пришла не одна выплата, реквизиты в профиле указанны!",[4361],1.000000,0.333333,-0.666667,"[4218, 4516, 4361, 4310, 4204, 4153, 4517, 1951, 3012, 4304]"


## 9. Test inference и `answer.csv`

`test.f` не участвовал в выборе. Берём score-матрицу победившего метода, возвращаем top-10 уникальных `article_id` и проверяем формат.

In [78]:
if "reranker" in best_method:
    reranker_model = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_NAME, revision=RERANKER_REVISION, local_files_only=True
    ).eval()
    test_candidates = candidate_article_indices(lexical_test, dense_test)
    reranker_test = reranker_scores(
        test["query_clean"], test_candidates, best_test_chunks
    )
    test_methods["cross-encoder reranker"] = reranker_test
    test_methods["RRF: 2×lexical + dense + 2×reranker"] = rrf(
        [lexical_test, dense_test, reranker_test], [2, 1, 2]
    )

test_predictions = rank_articles(test_methods[best_method], 10)
answer = test[["query_id"]].copy()
answer["answer"] = [" ".join(map(str, row)) for row in test_predictions]
answer_path = OUTPUT_DIR / "answer.csv"
answer.to_csv(answer_path, index=False)

saved = pd.read_csv(answer_path, dtype={"answer": "string"})
parsed = saved["answer"].str.split().map(lambda values: list(map(int, values)))
known_ids = set(articles["article_id"])
assert len(saved) == len(test)
assert saved.columns.tolist() == ["query_id", "answer"]
assert not saved.isna().any().any()
assert parsed.map(len).between(1, 10).all()
assert parsed.map(lambda values: len(values) == len(set(values))).all()
assert parsed.map(lambda values: set(values) <= known_ids).all()
assert saved["query_id"].tolist() == test["query_id"].tolist()

display(saved.head())
print(
    f"Сохранено: {answer_path}; строк: {len(saved)}; "
    f"победитель: {best_method}; 5-fold MAP@10: {best_map10:.6f}"
)

,query_id,answer
0,1,4361 4396 4308 4331 2964 1899 3565 4219 4205 1960
1,2,4009 1923 2408 2521 4407 3209 4403 4408 2802 3149
2,3,1909 4396 4329 4400 3419 4234 2831 4219 4362 4331
3,4,4400 4234 4219 3149 2646 4361 3006 2696 4331 4308
4,5,2698 4214 3128 4308 4388 4409 1747 4320 4302 3690


Сохранено: output/answer.csv; строк: 500; победитель: RRF: 2×lexical + dense; 5-fold MAP@10: 0.406026
